# Day 1 (Part 2): Tensor Gymnastics for Transformers
### Master the 5 Core Operations in Multi-Head Attention (MHA)

This notebook walks through the exact tensor shape transformations used in LLMs (GPT, Llama, Qwen).
Run these cells sequentially to build complete muscle memory for tensor dimensions.

In [1]:
import torch
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")

Active device: cuda
GPU: NVIDIA GeForce RTX 5070


## Drill 1: The Standard $(B, T, C)$ Anatomy
- **B** = Batch size (number of independent prompts/sequences)
- **T** = Sequence length / Time (number of tokens in the sequence)
- **C** = Channels / Embedding dimension (vector size per token, e.g. 768 or 4096)

In [12]:
B, T, C = 2, 4, 32 # Batch size is 2, 4 number of tokens in vocabulary, 32 channels/dimensions for the embeddings

x = torch.randn(B, T, C, device=device)
print(f"Tensort shape: {x.shape}")
print("Sample token vector (Sentence 0, token 0):", x[0][0][:6])

Tensort shape: torch.Size([2, 4, 32])
Sample token vector (Sentence 0, token 0): tensor([-0.3566,  1.6984, -0.0397, -0.4348, -0.2906, -3.2396], device='cuda:0')


## Drill 2: Splitting Heads with `.view()`
Instead of 1 large attention head of size $C$, we split into $n_{\text{head}}$ parallel heads, each with dimension $d_{\text{head}} = C / n_{\text{head}}$.

In [14]:
n_head = 4
head_dim = C // n_head

# Let's start with our Query tensor
Q = torch.randn(B, T, C, device=device)
Q_split = Q.view(B, T, n_head, head_dim)

print("Original Q:", Q.shape)
print("After .view():", Q_split.shape, "-> (B, T, n_head, head_dim)")

Original Q: torch.Size([2, 4, 32])
After .view(): torch.Size([2, 4, 4, 8]) -> (B, T, n_head, head_dim)


## Drill 3: The Transpose Trick & Contiguity
PyTorch `@` (matmul) multiplies the **last two dimensions**.
To calculate attention between all pairs of tokens $(T \times T)$ per head in parallel, we swap `SeqLen` and `NumHeads`.

In [15]:
# Swap dimension 1(T) and 2 (n_head): (B,T,nh,hs) -> (B,nh,T,hs)
Q_transposed = Q_split.transpose(1,2)
print("After transpose:", Q_transposed.shape, "-> (B, nh, T, hs)")
print("Is contiguous in memory?", Q_transposed.is_contiguous())

Q_contiguous = Q_transposed.contiguous()
print("After contiguous:", Q_contiguous.shape, "-> (B, nh, T, hs)")
print("Is contiguous in memory?", Q_contiguous.is_contiguous())

After transpose: torch.Size([2, 4, 4, 8]) -> (B, nh, T, hs)
Is contiguous in memory? False
After contiguous: torch.Size([2, 4, 4, 8]) -> (B, nh, T, hs)
Is contiguous in memory? True


## Drill 4: Attention Matrix Multiplication ($Q @ K^T$)
Multiply Queries with transposed Keys to get token-to-token scores $(T \times T)$.

In [ ]:
K = torch.randn(B, n_head, T, head_dim, device=device)
# Transpose last two dims: (B, nh, T, hs) -> (B, nh, hs, T)
K_t = K.transpose(-2, -1)

# Scaled Dot-Product scores: (B, nh, T, hs) @ (B, nh, hs, T) -> (B, nh, T, T)
scores = (Q_contiguous @ K_t) / (head_dim ** 0.5)
print("Attention Scores shape:", scores.shape)
print("Batch 0, Head 0 Scores matrix:\n", scores[0, 0].cpu().round(decimals=2))

## Drill 5: Causal Masking & Reassembly
Autoregressive models must not attend to future tokens. Mask with $-\infty$, apply softmax, multiply with Values $V$, and reassemble back to $(B, T, C)$.

In [ ]:
# 1. Lower-triangular mask
mask = torch.tril(torch.ones(T, T, device=device))

# 2. Mask future tokens with -inf
masked_scores = scores.masked_fill(mask == 0, float("-inf"))

# 3. Softmax
weights = F.softmax(masked_scores, dim=-1)
print("Attention weights (each row sums to 1.0):\n", weights[0, 0].cpu().round(decimals=3))

# 4. Multiply with Values V
V = torch.randn(B, n_head, T, head_dim, device=device)
out = weights @ V  # (B, nh, T, T) @ (B, nh, T, hs) -> (B, nh, T, hs)

# 5. Reassemble heads: (B, nh, T, hs) -> (B, T, nh, hs) -> (B, T, C)
out = out.transpose(1, 2).contiguous().view(B, T, n_head * head_dim)
print("Final MHA Output shape:", out.shape, "(Matches original (B, T, C)!)")

In [35]:
# 1. Lower-triangular mask
mask = torch.tril(torch.ones(T, T, device=device))

# 2. Mask future tokens with -inf
masked_scores = scores.masked_fill(mask == 0, float("-inf"))

# 3. softmax
weights = F.softmax(masked_scores, dim=-1)
print("Attention weights (each row sums to 1.0):\n", weights[0, 0].cpu().round(decimals=3))

# 4. Multiply with values V
V = torch.randn(B, n_head, T, head_dim, device=device)
out = weights @ V

# 5. Reassemble heads: (B, nh, T, hs) -> (B, T, nh, hs) -> (B, T, C)
out = out.transpose(1, 2).contiguous().view(B, T, n_head * head_dim)
print("Final MHA Output shape:", out.shape, "(Matches original (B, T, C)!)")

Attention weights (each row sums to 1.0):
 tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.6650, 0.3350, 0.0000, 0.0000],
        [0.5000, 0.4450, 0.0550, 0.0000],
        [0.2690, 0.1450, 0.4400, 0.1460]])
Final MHA Output shape: torch.Size([2, 4, 32]) (Matches original (B, T, C)!)
